# Análisis Diagnóstico: Comprobación de Hipótesis

En la fase **diagnóstica** de CRISP-DM buscamos ir más allá de describir los datos: queremos **comprobar, con evidencia estadística formal**, si existe una relación entre el ingreso anual, el trayecto diario y el acceso a puntos de recarga con la decisión de comprar un vehículo eléctrico (`Will_Buy_EV`).

Para respaldar la hipótesis planteada, se descompone en **cuatro pruebas**:

1. **H1a** — Ingreso anual vs. decisión de compra (prueba de dos muestras independientes)
2. **H1b** — Trayecto diario vs. decisión de compra (prueba de dos muestras independientes)
3. **H1c** — Acceso a carga en casa vs. decisión de compra (prueba de independencia de variables categóricas)
4. **H1 conjunta** — El conjunto {ingreso, trayecto diario, acceso a recarga} aporta información significativa para predecir la compra (prueba conjunta mediante regresión logística — esta es la hipótesis exacta planteada en el planteamiento del proyecto)

Las pruebas 1–3 sirven de respaldo individual (evidencia univariada) y la prueba 4 corresponde exactamente al enunciado de $H_0$ / $H_1$ del proyecto (evidencia multivariada conjunta).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

sns.set_style("whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

df = pd.read_csv("EV_Adoption_and_Range_Anxiety_Dataset.csv")  # ajustar ruta/nombre según tu notebook

print("Dimensiones:", df.shape)
df[["Annual_Income_USD", "Daily_Commute_km", "Home_Charging_Possible", "Will_Buy_EV"]].isna().sum()

**Nota de preprocesamiento:** `Annual_Income_USD` y `Daily_Commute_km` tienen valores faltantes (~1.8 % cada una). Para las pruebas de hipótesis se excluyen esos registros puntualmente (`dropna` por variable), en vez de eliminar toda la fila del dataset, para no perder observaciones válidas de las demás variables. Esto es coherente con la limpieza que se documenta en la sección de preprocesamiento.

## 1. Verificación de supuestos: ¿normalidad?

Antes de elegir la prueba estadística, se evalúa si `Annual_Income_USD` y `Daily_Commute_km` siguen una distribución normal, usando la **prueba de D'Agostino-Pearson** (`scipy.stats.normaltest`), preferida sobre Shapiro-Wilk cuando $n$ es grande (>5000).

In [ ]:
for col in ["Annual_Income_USD", "Daily_Commute_km"]:
    stat, p = stats.normaltest(df[col], nan_policy="omit")
    print(f"{col}: estadístico={stat:.2f}, p-valor={p:.2e}  -> {'NO normal' if p < 0.05 else 'normal'}")

Ambas variables **rechazan normalidad** ($p < 0.001$). Por lo tanto, en lugar de la prueba $t$ de Student clásica se usa su alternativa no paramétrica, la **prueba U de Mann-Whitney**, que compara las distribuciones de rangos entre los dos grupos (`Yes` / `No` en `Will_Buy_EV`) sin asumir normalidad.

## 2. Hipótesis H1a — Ingreso anual y decisión de compra

$H_0$: La distribución del ingreso anual es la misma entre quienes compran y quienes no compran un vehículo eléctrico.
$H_1$: La distribución del ingreso anual difiere entre ambos grupos.

$\alpha = 0.05$

In [ ]:
yes = df.loc[df["Will_Buy_EV"] == "Yes", "Annual_Income_USD"].dropna()
no  = df.loc[df["Will_Buy_EV"] == "No",  "Annual_Income_USD"].dropna()

u_stat, p_val = stats.mannwhitneyu(yes, no, alternative="two-sided")
r_rb = 1 - (2 * u_stat) / (len(yes) * len(no))  # correlación biserial de rangos (tamaño del efecto)

print(f"Ingreso promedio (Compra=Sí): ${yes.mean():,.0f}")
print(f"Ingreso promedio (Compra=No): ${no.mean():,.0f}")
print(f"U de Mann-Whitney = {u_stat:,.1f}, p-valor = {p_val:.2e}")
print(f"Tamaño del efecto (r biserial de rangos) = {r_rb:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x="Will_Buy_EV", y="Annual_Income_USD", ax=ax)
ax.set_title("Ingreso anual según decisión de compra de VE")
ax.set_xlabel("¿Comprará VE?")
ax.set_ylabel("Ingreso anual (USD)")
plt.tight_layout()
plt.show()

**Resultado:** $p < 0.001$, por lo que se **rechaza $H_0$**. El ingreso anual de los compradores (≈ \$98,000) es significativamente mayor que el de los no compradores (≈ \$82,700). El tamaño del efecto ($r \approx 0.26$) es de magnitud pequeña-a-moderada: la diferencia es estadísticamente significativa pero no extremadamente grande en términos prácticos, lo cual es esperable con $n=10{,}000$ (una muestra grande detecta diferencias pequeñas como significativas).

## 3. Hipótesis H1b — Trayecto diario y decisión de compra

$H_0$: La distribución del trayecto diario (km) es la misma entre compradores y no compradores.
$H_1$: La distribución del trayecto diario difiere entre ambos grupos.

$\alpha = 0.05$

In [ ]:
yes_c = df.loc[df["Will_Buy_EV"] == "Yes", "Daily_Commute_km"].dropna()
no_c  = df.loc[df["Will_Buy_EV"] == "No",  "Daily_Commute_km"].dropna()

u_stat_c, p_val_c = stats.mannwhitneyu(yes_c, no_c, alternative="two-sided")
r_rb_c = 1 - (2 * u_stat_c) / (len(yes_c) * len(no_c))

print(f"Trayecto promedio (Compra=Sí): {yes_c.mean():.1f} km")
print(f"Trayecto promedio (Compra=No): {no_c.mean():.1f} km")
print(f"U de Mann-Whitney = {u_stat_c:,.1f}, p-valor = {p_val_c:.4f}")
print(f"Tamaño del efecto (r biserial de rangos) = {r_rb_c:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x="Will_Buy_EV", y="Daily_Commute_km", ax=ax)
ax.set_title("Trayecto diario según decisión de compra de VE")
ax.set_xlabel("¿Comprará VE?")
ax.set_ylabel("Trayecto diario (km)")
plt.tight_layout()
plt.show()

**Resultado:** $p = 0.0076 < 0.05$, por lo que se **rechaza $H_0$**, aunque con menor margen que en el caso del ingreso. Los compradores recorren en promedio un poco menos (≈ 39.7 km) que los no compradores (≈ 41.4 km). El efecto es pequeño ($r \approx 0.03$), lo que sugiere que, aunque la diferencia es estadísticamente detectable dado el tamaño muestral, el trayecto diario por sí solo aporta poca capacidad discriminante — algo relevante para interpretar más adelante su peso relativo en el modelo predictivo.

## 4. Hipótesis H1c — Acceso a carga en casa y decisión de compra

$H_0$: `Home_Charging_Possible` y `Will_Buy_EV` son independientes (no hay asociación).
$H_1$: `Home_Charging_Possible` y `Will_Buy_EV` están asociadas.

Al ser ambas variables categóricas binarias, se usa la **prueba Chi-cuadrado de independencia** sobre la tabla de contingencia.

In [ ]:
tabla = pd.crosstab(df["Home_Charging_Possible"], df["Will_Buy_EV"])
print(tabla)

chi2, p_chi, dof, esperado = stats.chi2_contingency(tabla)
n = tabla.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))

print(f"\nChi2 = {chi2:.2f}, gl = {dof}, p-valor = {p_chi:.2e}")
print(f"V de Cramér (tamaño del efecto) = {cramers_v:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
(tabla.div(tabla.sum(axis=1), axis=0) * 100).plot(kind="bar", stacked=True, ax=ax)
ax.set_title("% que compra VE según acceso a carga en casa")
ax.set_xlabel("¿Carga en casa posible?")
ax.set_ylabel("% de compradores")
ax.legend(title="¿Comprará VE?")
plt.tight_layout()
plt.show()

**Resultado:** $\chi^2(1) = 73.10$, $p < 0.001$, por lo que se **rechaza $H_0$**: existe una asociación significativa entre tener posibilidad de carga en casa y la decisión de comprar un VE. Quienes sí pueden cargar en casa compran VE en mayor proporción (≈ 20.1 %) que quienes no (≈ 13.4 %). El efecto (V de Cramér ≈ 0.086) es pequeño en magnitud pero consistente con las otras pruebas.

## 5. Hipótesis conjunta $H_0$ / $H_1$ (regresión logística)

Esta es la hipótesis **tal como se formuló en el planteamiento del proyecto**:

> $H_0$: El conjunto de variables formado por el ingreso, el trayecto diario y el acceso a puntos de recarga **no aporta** información significativa para predecir si un cliente comprará un vehículo eléctrico.
> $H_1$: El conjunto de variables **sí aporta** información significativa.

Las tres pruebas anteriores (H1a, H1b, H1c) evalúan cada variable **por separado**. Para comprobar la hipótesis **conjunta** se ajusta un modelo de **regresión logística binaria** (`Will_Buy_EV` ~ ingreso + trayecto + acceso a carga) y se compara su verosimilitud contra un modelo nulo (solo el intercepto) mediante una **prueba de razón de verosimilitud (Likelihood Ratio Test, LRT)**:

$$LR = -2(\ell_{nulo} - \ell_{completo}) \sim \chi^2_{gl}$$

donde $gl$ = número de predictores agregados (3). Este es el procedimiento estadístico estándar para responder exactamente a la pregunta "¿este conjunto de variables aporta información significativa?", y además el modelo resultante puede reutilizarse como uno de los modelos de la sección predictiva.

In [ ]:
datos = df.dropna(subset=["Annual_Income_USD", "Daily_Commute_km", "Home_Charging_Possible", "Will_Buy_EV"]).copy()

datos["y"] = (datos["Will_Buy_EV"] == "Yes").astype(int)
datos["Home_Charging_Bin"] = (datos["Home_Charging_Possible"] == "Yes").astype(int)
datos["Income_k"] = datos["Annual_Income_USD"] / 1000  # escalado para coeficientes interpretables

X = sm.add_constant(datos[["Income_k", "Daily_Commute_km", "Home_Charging_Bin"]])
y = datos["y"]

modelo = sm.Logit(y, X).fit(disp=0)
print(modelo.summary())

In [ ]:
print(f"Prueba de razón de verosimilitud (LRT) = {modelo.llr:.2f}")
print(f"Grados de libertad = {int(modelo.df_model)}")
print(f"p-valor (LLR) = {modelo.llr_pvalue:.2e}")
print(f"Pseudo R² de McFadden = {modelo.prsquared:.4f}")

# Odds ratios por variable, para interpretación de negocio
odds_ratios = pd.DataFrame({
    "coef": modelo.params,
    "OR": np.exp(modelo.params),
    "p-valor": modelo.pvalues
})
odds_ratios

**Resultado:** $LR(3) = 384.29$, $p < 0.001$ (Pseudo $R^2$ de McFadden $= 0.043$). Se **rechaza $H_0$**: el conjunto {ingreso, trayecto diario, acceso a carga en casa} sí aporta información conjunta y estadísticamente significativa para predecir la compra de un vehículo eléctrico, respaldando $H_1$.

Además, cada coeficiente individual es significativo dentro del modelo conjunto ($p < 0.01$ en los tres casos):

- **Ingreso** (`Income_k`): OR ≈ e^0.0142 ≈ 1.014 por cada \$1,000 adicionales de ingreso anual → por cada \$10,000 extra, las probabilidades de compra aumentan ≈ 15 %, manteniendo lo demás constante.
- **Trayecto diario** (`Daily_Commute_km`): coeficiente negativo (−0.0031, $p=0.009$) → a mayor trayecto diario, ligeramente **menor** probabilidad de compra (compatible con mayor ansiedad por autonomía en trayectos largos).
- **Acceso a carga en casa** (`Home_Charging_Bin`): OR ≈ e^0.5154 ≈ 1.67 → tener posibilidad de carga en casa **casi duplica** las probabilidades de comprar un VE, manteniendo ingreso y trayecto constantes. Es la variable con mayor peso relativo de las tres.

El Pseudo $R^2$ modesto (0.043) indica que, aunque el aporte es estadísticamente significativo, estas tres variables **por sí solas explican una fracción limitada** de la variabilidad en la decisión de compra — es esperable que otras variables del dataset (nivel de ansiedad por autonomía, preocupación ambiental, subsidio disponible) aporten poder explicativo adicional, lo cual se explora en la sección predictiva del proyecto con modelos que incluyen más variables.

## 6. Conclusión del análisis diagnóstico

| Hipótesis | Prueba | Estadístico | p-valor | Decisión |
|---|---|---|---|---|
| H1a — Ingreso | Mann-Whitney U | U = 8,712,044.5 | < 0.001 | Se rechaza $H_0$ |
| H1b — Trayecto diario | Mann-Whitney U | U = 6,667,782.5 | 0.0076 | Se rechaza $H_0$ |
| H1c — Acceso a carga | Chi-cuadrado | $\chi^2(1)$ = 73.10 | < 0.001 | Se rechaza $H_0$ |
| **H1 conjunta** (planteamiento del proyecto) | LRT (regresión logística) | LR(3) = 384.29 | < 0.001 | **Se rechaza $H_0$ / se acepta $H_1$** |

Con un nivel de significancia $\alpha = 0.05$, la evidencia estadística —tanto univariada como multivariada— **respalda consistentemente $H_1$**: el ingreso, el trayecto diario y el acceso a puntos de recarga sí aportan información significativa para predecir si un cliente comprará un vehículo eléctrico. El acceso a carga en casa es, de las tres, la variable con mayor efecto relativo sobre la decisión de compra.